In [ ]:
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt

PLOT_DIR = "../docs/presentation/images/plots/"
DATA_DIR = "./data/"

In [ ]:
sns.set_theme(style="whitegrid", palette="Spectral")
sns.set_context("paper")

#### Utils

In [ ]:
def splitting(df: pl.DataFrame, col:str, type: str, grid: int, init: str):
    return df.select("step", col).with_columns(
        pl.lit(type).alias("type"), 
        pl.lit(grid).alias("grid"),
        pl.lit(init).alias("init"),
        pl.lit(col).alias("id")
    ).rename({col: "value"})

def renaming(df: pl.DataFrame, suffix: str, prefix: str):
    mapping = { 
        "shape_net": 
        {
            f"08-02_18-31-55_ShapeNet_NeMo_Rec_local - {suffix}": "dmtet_16_sphere",
            f"08-02_17-19-48_ShapeNet_NeMo_Rec_local - {suffix}": "flex_32_random",
            f"08-02_16-08-21_ShapeNet_NeMo_Rec_local - {suffix}": "flex_16_random",
            f"08-01_16-48-36_ShapeNet_NeMo_Rec_local - {suffix}": "dmtet_32_random",
            f"08-01_16-23-41_ShapeNet_NeMo_Rec_local - {suffix}": "dmtet_32_sphere",
            f"08-01_15-07-07_ShapeNet_NeMo_Rec_local - {suffix}": "flex_32_sphere",
            f"08-01_14-45-02_ShapeNet_NeMo_Rec_local - {suffix}": "dmtet_16_random",
            f"08-01_13-40-44_ShapeNet_NeMo_Rec_local - {suffix}": "flex_16_sphere",
            "Step": "step"
        },
        "co3d": 
        {
            f"08-04_12-10-50_CO3D_NeMo_Rec_local - {suffix}": "dmtet_32_random",
            f"08-04_11-24-50_CO3D_NeMo_Rec_local - {suffix}": "dmtet_32_sphere",
            f"08-04_10-53-57_CO3D_NeMo_Rec_local - {suffix}": "flex_32_random",
            f"08-03_23-34-45_CO3D_NeMo_Rec_local - {suffix}": "flex_16_random",
            "Step": "step"
        }
    }
    df = df.rename(mapping=mapping[prefix])

    cols = [
        "step",
        "dmtet_16_sphere",
        "flex_32_random",
        "flex_16_random",
        "dmtet_32_random",
        "dmtet_32_sphere",
        "flex_32_sphere",
        "dmtet_16_random",
        "flex_16_sphere",
    ]

    map = {
        i : {
            k[0]: k[1]
            for k in zip(["type", "grid", "init"], i.split("_"))
        }
        for i in cols[1:]
    }
    

    df = df.select(*cols)

    dfs = []
    for col in cols[1:]:
        vals = map[col]
        dfs.append(splitting(df, col, vals["type"], int(vals["grid"]), vals["init"]))

    df = pl.concat(dfs)
    return df

## ShapeNet 

In [ ]:
shape_net_train_loss = pl.read_csv(DATA_DIR + "shape_net_train_loss" + ".csv")
shape_net_val_iou = pl.read_csv(DATA_DIR + "shape_net_val_iou" + ".csv")
shape_net_val_loss = pl.read_csv(DATA_DIR + "shape_net_val_loss" + ".csv")
shape_net_val_psnr = pl.read_csv(DATA_DIR + "shape_net_val_psnr" + ".csv")
shape_net_val_rec_mask_mse = pl.read_csv(DATA_DIR + "shape_net_val_rec_mask_mse" + ".csv")
shape_net_val_rec_rgb_mse = pl.read_csv(DATA_DIR + "shape_net_val_rec_rgb_mse" + ".csv")
shape_net_test_iou = pl.read_csv(DATA_DIR + "shape_net_test_iou" + ".csv")
shape_net_test_psnr = pl.read_csv(DATA_DIR + "shape_net_test_psnr" + ".csv")



In [ ]:
shape_net_train_loss = renaming(shape_net_train_loss, "train/loss", "shape_net")
shape_net_val_iou = renaming(shape_net_val_iou, "val/shapenet/iou", "shape_net")
shape_net_val_loss = renaming(shape_net_val_loss, "val/shapenet/loss", "shape_net")
shape_net_val_psnr = renaming(shape_net_val_psnr, "val/shapenet/psnr", "shape_net")
shape_net_val_rec_mask_mse = renaming(shape_net_val_rec_mask_mse, "val/shapenet/rec_mask_mse", "shape_net")
shape_net_val_rec_rgb_mse = renaming(shape_net_val_rec_rgb_mse, "val/shapenet/rec_rgb_mse", "shape_net")
shape_net_test_iou = renaming(shape_net_test_iou, "test/shapenet/iou", "shape_net")
shape_net_test_psnr = renaming(shape_net_test_psnr, "test/shapenet/psnr", "shape_net")

#### Plots

In [ ]:
sns.lineplot(shape_net_train_loss, x="step", y="value", hue="id")
plt.title("Loss during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_train_loss" + ".png")

In [ ]:
sns.lineplot(shape_net_val_iou, x="step", y="value", hue="id", style="type")
plt.title("IOU on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_iou" + ".png")

In [ ]:
sns.lineplot(shape_net_val_iou, x="step", y="value", hue="init", style="type")
plt.title("IOU on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_iou_by_init_type" + ".png")

In [ ]:
sns.lineplot(shape_net_val_loss, x="step", y="value", hue="id", style="type")
plt.title("Loss on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_loss" + ".png")

In [ ]:
sns.lineplot(shape_net_val_loss, x="step", y="value", hue="init", style="type")
plt.title("Loss on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_loss_by_init_type" + ".png")

In [ ]:
sns.lineplot(shape_net_val_psnr, x="step", y="value", hue="id", style="type")
plt.title("PSNR on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_psnr" + ".png")

In [ ]:
sns.lineplot(shape_net_val_psnr, x="step", y="value", hue="init", style="type")
plt.title("PSNR on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_psnr_by_init_type" + ".png")

In [ ]:
sns.lineplot(shape_net_val_rec_mask_mse, x="step", y="value", hue="id", style="type")
plt.title("MSE of Mask on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_rec_mask_mse" + ".png")

In [ ]:
sns.lineplot(shape_net_val_rec_rgb_mse, x="step", y="value", hue="id", style="type")
plt.title("MSE of RGB on Validation Set during Training (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_val_rec_rgb_mse" + ".png")

In [ ]:
sns.barplot(shape_net_test_iou, x="type", y="value", hue="type")
plt.title("IoU at Test time (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_test_iou_by_type" + ".png")

In [ ]:
sns.barplot(shape_net_test_iou, x="id", y="value", hue="id")
plt.title("IoU at Test time (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_test_iou_by_id" + ".png")

In [ ]:
sns.barplot(shape_net_test_iou, x="init", y="value", hue="type")
plt.title("IoU at Test time (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_test_iou_by_init_type" + ".png")

In [ ]:
sns.barplot(shape_net_test_psnr, x="type", y="value", hue="type")
plt.title("PSNR at Test time (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_test_psnr_by_type" + ".png")

In [ ]:
sns.barplot(shape_net_test_psnr, x="id", y="value", hue="id")
plt.title("PSNR at Test time (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_test_psnr_by_id" + ".png")

In [ ]:
sns.barplot(shape_net_test_psnr, x="init", y="value", hue="type")
plt.title("PSNR at Test time (ShapeNet)")
plt.savefig(PLOT_DIR + "shape_net_test_psnr_by_init_type" + ".png")

## CO3D

In [ ]:
co3d_train_loss = pl.read_csv(DATA_DIR + "co3d_train_loss" + ".csv")
co3d_val_iou = pl.read_csv(DATA_DIR + "co3d_val_iou" + ".csv")
co3d_val_loss = pl.read_csv(DATA_DIR + "co3d_val_loss" + ".csv")
co3d_val_psnr = pl.read_csv(DATA_DIR + "co3d_val_psnr" + ".csv")
co3d_val_rec_mask_mse = pl.read_csv(DATA_DIR + "co3d_val_rec_mask_mse" + ".csv")
co3d_val_rec_rgb_mse = pl.read_csv(DATA_DIR + "co3d_val_rec_rgb_mse" + ".csv")
co3d_test_iou = pl.read_csv(DATA_DIR + "co3d_test_iou" + ".csv")
co3d_test_psnr = pl.read_csv(DATA_DIR + "co3d_test_psnr" + ".csv")

In [ ]:
co3d_train_loss = renaming(co3d_train_loss, "train/loss", "co3d")
co3d_val_iou = renaming(co3d_val_iou, "val/co3d_no_zsp_1s_labeled_ref/iou", "co3d")
co3d_val_loss = renaming(co3d_val_loss, "val/co3d_no_zsp_1s_labeled_ref/loss", "co3d")
co3d_val_psnr = renaming(co3d_val_psnr, "val/co3d_no_zsp_1s_labeled_ref/psnr", "co3d")
co3d_val_rec_mask_mse = renaming(co3d_val_rec_mask_mse, "val/co3d_no_zsp_1s_labeled_ref/rec_mask_mse", "co3d")
co3d_val_rec_rgb_mse = renaming(co3d_val_rec_rgb_mse, "val/co3d_no_zsp_1s_labeled_ref/rec_rgb_mse", "co3d")
co3d_test_iou = renaming(co3d_test_iou, "test/co3d_no_zsp_1s_labeled_ref/iou", "co3d")
co3d_test_psnr = renaming(co3d_test_psnr, "test/co3d_no_zsp_1s_labeled_ref/psnr", "co3d")

In [ ]:
sns.lineplot(co3d_train_loss, x="step", y="value", hue="id")
plt.title("Loss during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_train_loss" + ".png")

In [ ]:
sns.lineplot(co3d_val_iou, x="step", y="value", hue="id", style="type")
plt.title("IOU on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_iou" + ".png")

In [ ]:
sns.lineplot(co3d_val_iou, x="step", y="value", hue="init", style="type")
plt.title("IOU on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_iou_by_init_type" + ".png")

In [ ]:
sns.lineplot(co3d_val_loss, x="step", y="value", hue="id", style="type")
plt.title("Loss on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_loss" + ".png")

In [ ]:
sns.lineplot(co3d_val_loss, x="step", y="value", hue="init", style="type")
plt.title("Loss on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_loss_by_init_type" + ".png")

In [ ]:
sns.lineplot(co3d_val_psnr, x="step", y="value", hue="id", style="type")
plt.title("PSNR on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_psnr" + ".png")

In [ ]:
sns.lineplot(co3d_val_psnr, x="step", y="value", hue="init", style="type")
plt.title("PSNR on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_psnr_by_init_type" + ".png")

In [ ]:
sns.lineplot(co3d_val_rec_mask_mse, x="step", y="value", hue="id", style="type")
plt.title("MSE of Mask on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_rec_mask_mse" + ".png")

In [ ]:
sns.lineplot(co3d_val_rec_rgb_mse, x="step", y="value", hue="id", style="type")
plt.title("MSE of RGB on Validation Set during Training (CO3D)")
plt.savefig(PLOT_DIR + "co3d_val_rec_rgb_mse" + ".png")

In [ ]:
sns.barplot(co3d_test_iou, x="type", y="value", hue="type")
plt.title("IoU at Test time (CO3D)")
plt.savefig(PLOT_DIR + "co3d_test_iou_by_type" + ".png")

In [ ]:
sns.barplot(co3d_test_iou, x="id", y="value", hue="id")
plt.title("IoU at Test time (CO3D)")
plt.savefig(PLOT_DIR + "co3d_test_iou_by_id" + ".png")

In [ ]:
sns.barplot(co3d_test_iou, x="init", y="value", hue="type")
plt.title("IoU at Test time (CO3D)")
plt.savefig(PLOT_DIR + "co3d_test_iou_by_init_type" + ".png")

In [ ]:
sns.barplot(co3d_test_psnr, x="type", y="value", hue="type")
plt.title("PSNR at Test time (CO3D)")
plt.savefig(PLOT_DIR + "co3d_test_psnr_by_type" + ".png")

In [ ]:
sns.barplot(co3d_test_psnr, x="id", y="value", hue="id")
plt.title("PSNR at Test time (CO3D)")
plt.savefig(PLOT_DIR + "co3d_test_psnr_by_id" + ".png")

In [ ]:
sns.barplot(co3d_test_psnr, x="init", y="value", hue="type")
plt.title("PSNR at Test time (CO3D)")
plt.savefig(PLOT_DIR + "co3d_test_psnr_by_init_type" + ".png")